# Image to Paragraph
___

In [6]:
#Library Import
import os
import pickle
import numpy as np
import clip
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import words, stopwords
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from FlagEmbedding import FlagModel
from search_code import *
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

#Download from nltk
nltk.download('words')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\Patrick\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Patrick\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Patrick\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Patrick\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

#### Code

In [11]:
def image_keywords(image_path, pickle_folder="Pickle", top_k=10):
    # Load the model
    clip_device = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model, clip_preprocess = clip.load("ViT-B/32", device=clip_device)

    # Load and preprocess the image
    image = clip_preprocess(Image.open(image_path)).unsqueeze(0).to(clip_device)

    # Load the preprocessed text features from the pickle file
    pickle_file = os.path.join(pickle_folder, "Keywords_Token.pkl")
    with open(pickle_file, "rb") as f:
        all_keywords_features = pickle.load(f)

    # Ensure all_keywords_features are on the same device as the image features
    all_keywords_features = all_keywords_features.to(clip_device)
    image_features = clip_model.encode_image(image).detach()
    image_features /= image_features.norm(dim=-1, keepdim=True)

    # Calculate similarity between image and the pre-loaded keyword features
    similarities = (image_features @ all_keywords_features.T).squeeze(0)

    # Sort and retrieve the top `top_k` keywords
    top_indices = similarities.topk(top_k).indices

    # Load possible_keywords from the "All Keywords.pkl" file in the Pickle folder
    all_keywords_pickle_path = os.path.join(pickle_folder, "All Keywords.pkl")
    with open(all_keywords_pickle_path, "rb") as f:
        possible_keywords = pickle.load(f)

    # Get the top keywords based on the indices
    best_keywords = [possible_keywords[idx] for idx in top_indices.cpu().numpy()]

    return best_keywords

def search_paragraphs(image_path, n):
    #Get top keywords from the image
    image_keywords_result = image_keywords(image_path)
    print("Top Keywords from Image:", image_keywords_result)

    #Use the keywords obtained from the image as the query text
    query_text = " ".join(image_keywords_result)
    print(f"Query Text: {query_text}")

    #Choose Library
    print("Choose Library:")
    print("1. Commodification")
    print("2. Familiar")
    print("3. Digital Culture")
    print("4. Uncanny")
    folder_choice = input("Enter Library Number: ")

    folder_map = {
        "1": "Commodification",
        "2": "Familiar",
        "3": "Digital Culture",
        "4": "Uncanny"
    }

    folder = folder_map.get(folder_choice, None)
    if folder is None:
        print("Invalid choice.")
        return

    # Choose Library Code
    paths = {
        "windows": os.path.join("..","Workshop1","Pickle", folder),
        "macos": os.path.join("..","Workshop1","Pickle", folder)
    }

    #Pre-Process Query Texts
    processor = TextProcessor(paths)
    processed_query = processor.process_query(query_text)  # Use query_text directly
    print(f"Processed Query: {processed_query}")

    #Choose the search type
    print("Choose the search type:")
    print("1. TF-IDF_search")
    print("2. SVD_search")
    print("3. w2v_search")
    print("4. LLM_search")
    search_choice = input("Enter Search Engine: ")

    #Load the Library
    processor.load_database()

    #Choose the "Search Engine" code
    if search_choice == "1":
        #TF-IDF search
        tfidf_search(processed_query, n, processor)
    elif search_choice == "2":
        #SVD search
        svd_search(processed_query, n, processor)
    elif search_choice == "3":
        #Word2Vec search
        word2vec_search(processed_query, n, processor)
    elif search_choice == "4":
        #LLM search
        LLM_search(processed_query, n, processor)
    else:
        print("Invalid choice.")

#### Execute Code

In [13]:
image_path = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_SkillsClasses/Workshop2/Datasets/Fossil/Are Neanderthals the same species as us_ _ Natural History Museum.jpg"
search_paragraphs(image_path, n=1)

Top Keywords from Image: ['minuscul', 'mees', 'majesti', 'abuna', 'tetraspor', 'irrecognit', 'flector', 'alef', 'timeli', 'dropout']
Query Text: minuscul mees majesti abuna tetraspor irrecognit flector alef timeli dropout
Choose Library:
1. Commodification
2. Familiar
3. Digital Culture
4. Uncanny


Enter Library Number:  4


Processed Query: abuna flector alef dropout
Choose the search type:
1. TF-IDF_search
2. SVD_search
3. w2v_search
4. LLM_search


Enter Search Engine:  4


Number of .pkl files: 90
Processed Query: abuna flector alef dropout


pre tokenize: 100%|█████████████████████████████████████████████████████████████████| 601/601 [00:05<00:00, 106.79it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████████████████████████████████████████████████████| 601/601 [02:37<00:00,  3.81it/s]


LLM Search Results:

Rank 1:
Similarity Score: 0.5508,
DOCUMENT_____
TEXT: 'Avecdesmotsordinaires,onn’“épatepaslebourgeois,”nile“peuple.”Ilfaut desmotsextraordinaires.Enfait,paradoxalement,lemondedel’imageest dominéparlesmots.Laphoton’estriensanslalégendequiditcequ’ilfaut lire—legendum—,c’est-à-dire,biensouvent,deslégendes,quifontvoir n’importequoi.Nommer,onlesait,c’estfairevoir,c’estcréer,porterà l’existence.Etlesmotspeuventfairedesravages:islam,islamique,isla- miste—lefoulardest-ilislamiqueouislamiste?Ets’ils’agissaitsimplement d’unfichu,sansplus?Ilm’arrived’avoirenviedereprendrechaquemotdes présentateursquiparlentsouventàlalégère,sansavoirlamoindreidéedela difficultéetdelagravitédecequ’ilsévoquentetdesresponsabilitésqu’ils encourentenlesévoquant,devantdesmilliersdetéléspectateurs,sansles comprendreetsanscomprendrequ’ilsnelescomprennentpas.Parcequeces motsfontdeschoses,créentdesfantasmes,despeurs,desphobiesou,simple- ment,desreprésentationsfausses.(1996,19)'
LINE: 804
BOOK: 'State Po

In [14]:
image_path = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_Thesis_Project\JPG\1-3-scaled.jpg"
search_paragraphs(image_path, n=1)

Top Keywords from Image: ['radiolog', 'unenerget', 'arguabl', 'marzipan', 'frow', 'diligentia', 'kugel', 'blacker', 'fissur', 'angekok']
Query Text: radiolog unenerget arguabl marzipan frow diligentia kugel blacker fissur angekok
Choose Library:
1. Commodification
2. Familiar
3. Digital Culture
4. Uncanny


Enter Library Number:  3


Processed Query: marzipan frow diligentia kugel blacker angekok
Choose the search type:
1. TF-IDF_search
2. SVD_search
3. w2v_search
4. LLM_search


Enter Search Engine:  3


Number of .pkl files: 89

Processed Query for Word2Vec: marzipan frow diligentia kugel blacker angekok
Word2Vec Search Results: 

Rank 1: 
Similarity Score: 0.9327,
DOCUMENT_____
TEXT: ' Atlu. Tiga. Taru. Talu. Telo. 4 Ampat. Apat. Apat. Ampat. Apat. Ampat. Apat. 5 Lima. Lima. Lima. Lima. Rima. Lima. Limo. 6 Anam. Anim. Anam. Anam. Unum. Anam. Anam. 7 Tujoh. Pito. Pitu. Tujoh. Ijo. Tujoh.'
LINE: 3191
BOOK: 'pg38081-images'
KEYWORD: 'rima anam anam anam anam anam'



In [15]:
image_path = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_Thesis_Project\JPG\1-3-scaled.jpg"
search_paragraphs(image_path, n=1)

Top Keywords from Image: ['radiolog', 'unenerget', 'arguabl', 'marzipan', 'frow', 'diligentia', 'kugel', 'blacker', 'fissur', 'angekok']
Query Text: radiolog unenerget arguabl marzipan frow diligentia kugel blacker fissur angekok
Choose Library:
1. Commodification
2. Familiar
3. Digital Culture
4. Uncanny


Enter Library Number:  3


Processed Query: marzipan frow diligentia kugel blacker angekok
Choose the search type:
1. TF-IDF_search
2. SVD_search
3. w2v_search
4. LLM_search


Enter Search Engine:  1


Number of .pkl files: 89

Processed Query: marzipan frow diligentia kugel blacker angekok
Number of .pkl files: 89
TF-IDF Matrix Shape: (411918, 31777)
TF-IDF Search Results: 

Rank 1: 
Similarity Score: 0.4891,
DOCUMENT_____
TEXT: ' Mañana salgo de caza, y la tornaq me guiará. Luego vino el angekok, el hechicero de la aldea, y Kotuko refirió el mismo cuento por segunda vez. No perdió en lo más mínimo al ser repetido. —Sigue á los tornait (los espíritus de las piedras) y ellos volverán á darte comida, dijo el angekok.'
LINE: 2059
BOOK: 'pg69552-images'
KEYWORD: 'caza vino angekok ser la angekok'
Top Terms: angekok, vino, caza, ser, la, zygot, footer, footbal, footag, foot



In [16]:
image_path = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_Thesis_Project\JPG\1-3-scaled.jpg"
search_paragraphs(image_path, n=1)

Top Keywords from Image: ['radiolog', 'unenerget', 'arguabl', 'marzipan', 'frow', 'diligentia', 'kugel', 'blacker', 'fissur', 'angekok']
Query Text: radiolog unenerget arguabl marzipan frow diligentia kugel blacker fissur angekok
Choose Library:
1. Commodification
2. Familiar
3. Digital Culture
4. Uncanny


Enter Library Number:  3


Processed Query: marzipan frow diligentia kugel blacker angekok
Choose the search type:
1. TF-IDF_search
2. SVD_search
3. w2v_search
4. LLM_search


Enter Search Engine:  2


Number of .pkl files: 89

Processed Query for SVD: marzipan frow diligentia kugel blacker angekok
Number of .pkl files: 89
TF-IDF Matrix Shape: (411918, 31777)
SVD Search Results: 

Rank 1: 
Similarity Score: 0.6342,
DOCUMENT_____
TEXT: ', University of Virginia, 1817-26, plan (from Kimball, Thomas Jefferson) 83 13 Isaiah Rogers: Boston, Tremont House, 1828-9, plan (from Eliot, A Description of the Tremont House) 87 14 H.-P.-F. Labrouste: Paris, Bibliothèque Sainte-Geneviève, (1839), 1843-50, section (from Allgemeine Bauzeitung, 1851, plate 386) 125 15 J.'
LINE: 15
BOOK: 'pg70079-images'
KEYWORD: 'univers plan boston hous plan descript hous section plate'



In [17]:
image_path = r"E:\OneDrive\Documents\007-Study Life\001-Urban Design\RC11_Thesis_Project\JPG\1-3-scaled.jpg"
search_paragraphs(image_path, n=1)

Top Keywords from Image: ['radiolog', 'unenerget', 'arguabl', 'marzipan', 'frow', 'diligentia', 'kugel', 'blacker', 'fissur', 'angekok']
Query Text: radiolog unenerget arguabl marzipan frow diligentia kugel blacker fissur angekok
Choose Library:
1. Commodification
2. Familiar
3. Digital Culture
4. Uncanny


Enter Library Number:  3


Processed Query: marzipan frow diligentia kugel blacker angekok
Choose the search type:
1. TF-IDF_search
2. SVD_search
3. w2v_search
4. LLM_search


Enter Search Engine:  4


Number of .pkl files: 89
Processed Query: marzipan frow diligentia kugel blacker angekok


pre tokenize: 100%|█████████████████████████████████████████████████████████████████| 805/805 [00:07<00:00, 106.78it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████████████████████████████████████████████████████| 805/805 [03:23<00:00,  3.95it/s]


LLM Search Results:

Rank 1:
Similarity Score: 0.6064,
DOCUMENT_____
TEXT: ' bête noire (lit. a black beast,) (bet nwȧr´), a bugbear. beurre (bûr).—Butter. beurre fraîs (bûr frā).—Fresh (unsalted) butter. beurre lié (bûr lē-ā´).—Dutch sauce with less butter than usual. beurre noir (bûr nwär).'
LINE: 13166
BOOK: 'pg48661-images'
KEYWORD: 'lit black beast bet bugbear bur butter bur fra fresh unsalt butter lie bur dutch sauc less butter usual noir bur'

